

# Install dependancies

In [ ]:
%%shell
apt-get update
apt-get install -y build-essential python3-dev automake cmake git flex bison libglib2.0-dev libpixman-1-dev python3-setuptools cargo libgtk-3-dev
# try to install llvm-18 and install the distro default if that fails
apt-get install -y lld-18 llvm-18 llvm-18-dev clang-18 || sudo apt-get install -y lld llvm llvm-dev clang
apt-get install -y gcc-$(gcc --version|head -n1|sed 's/\..*//'|sed 's/.* //')-plugin-dev libstdc++-$(gcc --version|head -n1|sed 's/\..*//'|sed 's/.* //')-dev
apt-get install -y meson ninja-build # for QEMU mode
apt-get install -y cpio libcapstone-dev # for Nyx mode
apt-get install -y wget curl # for Frida mode
apt-get install -y python3-pip # for Unicorn mode

#update rustc
apt remove rustc
curl https://sh.rustup.rs -sSf | sh -s -- -y
. "$HOME/.cargo/env"
rustc --version


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,742 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,468 kB]
Ge

# Build AFLplusplus

### Install dependancies for python bindings and create a virtual environment

In [ ]:
%%shell
cd /content
pip install virtualenv
virtualenv fuzzingenv
source fuzzingenv/bin/activate
pip install setuptools
pip install pyelftools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 33.6 MB/s eta 0:00:00
created virtual environment CPython3.12.12.final.0-64 in 457ms
  creator CPython3Posix(dest=/content/fuzzingenv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==25.3
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.0 MB/s  0:00:00


### update rust and install afl++

In [ ]:

%%shell
cd /content
. "$HOME/.cargo/env"
source fuzzingenv/bin/activate
git clone https://github.com/AFLplusplus/AFLplusplus
cd AFLplusplus
git submodule update --init
make clean
make distrib NO_NYX=1 NO_CORESIGHT=1 NO_QEMU=1 NO_FRIDA=1
make install
# disable the /proc/sys/kernel/core_pattern check
# export AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES=1

fatal: destination path 'AFLplusplus' already exists and is not an empty directory.
[*] Compiling AFL++ for OS Linux on ARCH x86_64
[+] ZLIB detected
rm -rf afl-fuzz afl-showmap afl-tmin afl-gotcpu afl-analyze afl-cmin afl-fuzz-document as afl-as afl-g++ afl-clang afl-clang++ *.o src/*.o *~ a.out core core.[1-9][0-9]* *.stackdump .test .test1 .test2 test-instr .test-instr0 .test-instr1 afl-cs-proxy afl-qemu-trace afl-gcc-fast afl-g++-fast ld *.so *.8 test/unittests/*.o test/unittests/unit_maybe_alloc test/unittests/preallocable .afl-* afl-gcc afl-g++ afl-clang afl-clang++ test/unittests/unit_hash test/unittests/unit_rand *.dSYM lib*.a
make -f GNUmakefile.llvm clean
make[1]: Entering directory '/content/AFLplusplus'
[+] llvm_mode detected llvm 12+, enabling afl-lto LTO implementation
rm -f *.o *.so *~ a.out core core.[1-9][0-9]* .test2 test-instr .test-instr0 .test-instr1 *.dwo
rm -f ./afl-cc ./afl-compiler-rt.o ./afl-compiler-rt-32.o ./afl-compiler-rt-64.o ./afl-llvm-pass.so ./afl-llvm

# Test afl-fuzz binary is working correclty

In [ ]:
%env AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES=1
!afl-fuzz

env: AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES=1
[+] Enabled environment variable AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES with value 1
[!] WARNING: Potentially mistyped AFL environment variable: USE_AUTH_EPHEM=1, did you mean AFL_USE_AUTH_EPHEM=1?
afl-fuzz++4.36a based on afl by Michal Zalewski and a large online community

afl-fuzz [ options ] -- /path/to/fuzzed_app [ ... ]

Required parameters:
  -i dir        - input directory with test cases (or '-' to resume, also see 
                  AFL_AUTORESUME)
  -o dir        - output directory for fuzzer findings

Execution control settings:
  -P strategy   - set fix mutation strategy: explore (focus on new coverage),
                  exploit (focus on triggering crashes). You can also set a
                  number of seconds after without any finds it switches to
                  exploit mode, and back on new coverage (default: 1000)
  -p schedule   - power schedules compute a seed's performance score:
                  explore(default), f

 # Build a binary with AFLplusplus instrumentation

### Mount the drive as a local filesystem

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir("/content/drive/My Drive/Colab Notebooks/fuzzing")
!ls

Mounted at /content/drive
 dict			  harness.py	 Makefile   rustup.sh
 fuzzingenv		  input_basic	 out	    vulnerable-arm
'Fuzzing handson.ipynb'   input_corpus	 output     vulnerable.c


### Build the binary with afl-gcc to inject instrumentation

In [ ]:
!make clean
!make

rm -f vulnerable
rm -f vulnerable-arm
afl-gcc-fast -g -w  vulnerable.c -o vulnerable 	
afl-cc++4.36a by Michal Zalewski, Laszlo Szekeres, Marc Heuse - mode: GCC_PLUGIN-DEFAULT
afl-gcc-pass ++4.36a by <oliva@adacore.com>
[*] Inline instrumentation at ratio of 100% in non-hardened mode.
[+] Instrumented 14 locations (non-hardened mode, inline, ratio 100%).


### Test the binary with a random input

In [ ]:
!echo "Hello" | ./vulnerable

### Check the binary crash with the magic inputs

In [ ]:
!echo "foo!" | ./vulnerable

one
two
three
four
/bin/bash: line 1: 64189 Done                    echo "foo!"
     64190 Aborted                 (core dumped) | ./vulnerable


In [ ]:
!echo "GOODFUZZER" | ./vulnerable

one
two
/bin/bash: line 1: 64200 Done                    echo "GOODFUZZER"
     64201 Aborted                 (core dumped) | ./vulnerable


# Start fuzzing with basic inputs

In [ ]:
!rm -r ./output
!AFL_NO_UI=1 afl-fuzz -i ./input_basic -o ./output ./vulnerable

[+] Enabled environment variable AFL_NO_UI with value 1
[+] Enabled environment variable AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES with value 1
[!] WARNING: Potentially mistyped AFL environment variable: USE_AUTH_EPHEM=1, did you mean AFL_USE_AUTH_EPHEM=1?
afl-fuzz++4.36a based on afl by Michal Zalewski and a large online community
[+] AFL++ is maintained by Marc "van Hauser" Heuse, Dominik Maier, Andrea Fioraldi and Heiko "hexcoder" Eißfeldt
[+] AFL++ is open source, get it at https://github.com/AFLplusplus/AFLplusplus
[+] NOTE: AFL++ >= v3 has changed defaults and behaviours - see README.md
[+] No -M/-S set, autoconfiguring for "-S default"
[*] Getting to work...
[+] Using exploration-based constant power schedule (EXPLORE)
[+] Enabled testcache with 50 MB
[+] Generating fuzz data with a length of min=1 max=1048576
[*] Checking core_pattern...

[-] Your system is configured to send core dump notifications to an
    external utility. This will cause issues: there will be an extended delay

Show our findings : list the elements in the **queue** folder

These are the inputs that improved the code coverage

In [ ]:
!ls ./output/default/queue/
!find ./output/default/queue -name id* -exec cat {} \; -exec echo "" \;

id:000000,time:0,execs:0,orig:inpu1.txt
id:000001,src:000000,time:2451,execs:851,op:havoc,rep:3,+cov
id:000002,src:000001,time:20902,execs:7109,op:havoc,rep:2,+cov
id:000003,src:000002,time:22952,execs:7772,op:havoc,rep:2,+cov
rand
f@d
fo�d
foo 


Show our findings : list the elements in the **crash** folder

These are the inputs that caused a crash in the binary

In [ ]:
!ls ./output/default/crashes/
!find ./output/default/crashes -name id* -exec cat {} \; -exec echo "" \;

# Use better input corpus

In this cas the input corpus already passes 3 out of the 4 checks required to reach the crash.

The fuzzer only needs to find one addicitonal byte value.

In [ ]:
!rm -r ./output
!AFL_NO_UI=1 afl-fuzz -i ./input_corpus -o ./output ./vulnerable

[+] Enabled environment variable AFL_NO_UI with value 1
[+] Enabled environment variable AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES with value 1
[!] WARNING: Potentially mistyped AFL environment variable: USE_AUTH_EPHEM=1, did you mean AFL_USE_AUTH_EPHEM=1?
afl-fuzz++4.36a based on afl by Michal Zalewski and a large online community
[+] AFL++ is maintained by Marc "van Hauser" Heuse, Dominik Maier, Andrea Fioraldi and Heiko "hexcoder" Eißfeldt
[+] AFL++ is open source, get it at https://github.com/AFLplusplus/AFLplusplus
[+] NOTE: AFL++ >= v3 has changed defaults and behaviours - see README.md
[+] No -M/-S set, autoconfiguring for "-S default"
[*] Getting to work...
[+] Using exploration-based constant power schedule (EXPLORE)
[+] Enabled testcache with 50 MB
[+] Generating fuzz data with a length of min=1 max=1048576
[*] Checking core_pattern...

[-] Your system is configured to send core dump notifications to an
    external utility. This will cause issues: there will be an extended delay

Show our findings : list the elements in the **queue** folder

These are the inputs that improved the code coverage

In [ ]:
!ls ./output/default/queue/
!find ./output/default/queue -name id* -exec cat {} \; -exec echo "" \;

id:000000,time:0,execs:0,orig:input1.txt
id:000001,src:000000,time:25,execs:9,op:quick,pos:0,+cov
id:000002,src:000000,time:54,execs:17,op:quick,pos:1,+cov
id:000003,src:000000,time:84,execs:25,op:quick,pos:2,+cov

foo.
�oo.
fo.
fo�.


Show our findings : list the elements in the **crash** folder

These are the inputs that caused a crash in the binary

In [ ]:
!ls ./output/default/crashes/
!find ./output/default/crashes -name id* -exec cat {} \; -exec echo "" \;

id:000000,sig:06,src:000000,time:1736,execs:543,op:havoc,rep:1	README.txt
foo!


# Use a dictionnary

The dictionary dict.dct contains various keywords. Keywords are inserted by the fuzzer during the mutations.

A combination of 2 keywords is required in the input to reach the crash.

In [ ]:
!rm -r ./output
!AFL_NO_UI=1 afl-fuzz -x dict/dict.dct -i ./input_corpus -o ./output ./vulnerable

[+] Enabled environment variable AFL_NO_UI with value 1
[+] Enabled environment variable AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES with value 1
[!] WARNING: Potentially mistyped AFL environment variable: USE_AUTH_EPHEM=1, did you mean AFL_USE_AUTH_EPHEM=1?
afl-fuzz++4.36a based on afl by Michal Zalewski and a large online community
[+] AFL++ is maintained by Marc "van Hauser" Heuse, Dominik Maier, Andrea Fioraldi and Heiko "hexcoder" Eißfeldt
[+] AFL++ is open source, get it at https://github.com/AFLplusplus/AFLplusplus
[+] NOTE: AFL++ >= v3 has changed defaults and behaviours - see README.md
[+] No -M/-S set, autoconfiguring for "-S default"
[*] Getting to work...
[+] Using exploration-based constant power schedule (EXPLORE)
[+] Enabled testcache with 50 MB
[+] Generating fuzz data with a length of min=1 max=1048576
[*] Checking core_pattern...

[-] Your system is configured to send core dump notifications to an
    external utility. This will cause issues: there will be an extended delay

Show our findings : list the elements in the **queue** folder

These are the inputs that improved the code coverage

In [ ]:
!ls ./output/default/queue/
!find ./output/default/queue -name id* -exec cat {} \; -exec echo "" \;

id:000000,time:0,execs:0,orig:input1.txt
id:000001,src:000000,time:19,execs:9,op:quick,pos:0,+cov
id:000002,src:000000,time:40,execs:17,op:quick,pos:1,+cov
id:000003,src:000000,time:62,execs:25,op:quick,pos:2,+cov
id:000004,src:000000,time:1283,execs:431,op:ext_UO,pos:0,+cov

foo.
Qoo.
f�o.
fo�.
GOOD


[link text](https://)Show our findings : list the elements in the **crash** folder

These are the inputs that caused a crash in the binary

In [ ]:
!ls ./output/default/crashes/
!find ./output/default/crashes -name id* -exec cat {} \; -exec echo "" \;

id:000000,sig:06,src:000000,time:1679,execs:615,op:havoc,rep:1	 README.txt
id:000001,sig:06,src:000004,time:2755,execs:1003,op:havoc,rep:1
foo!.
GOODFUZZER


Replay our crash test cases

In [ ]:
!(cat ./output/default/crashes/id:000001,sig:06,src:000003,time:38067,execs:13913,op:havoc,rep:55 ; echo "") | ./vulnerable

cat: './output/default/crashes/id:000001,sig:06,src:000003,time:38067,execs:13913,op:havoc,rep:55': No such file or directory


# Fuzz an arm binary

### Install the cross compiler

In [ ]:
!apt install gcc-arm-linux-gnueabihf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  libstd-rust-1.75 libstd-rust-dev
Use 'apt autoremove' to remove them.
The following additional packages will be installed:
  binutils-arm-linux-gnueabihf cpp-11-arm-linux-gnueabihf
  cpp-arm-linux-gnueabihf gcc-11-arm-linux-gnueabihf
  gcc-11-arm-linux-gnueabihf-base gcc-11-cross-base gcc-12-cross-base
  libasan6-armhf-cross libatomic1-armhf-cross libc6-armhf-cross
  libc6-dev-armhf-cross libgcc-11-dev-armhf-cross libgcc-s1-armhf-cross
  libgomp1-armhf-cross libstdc++6-armhf-cross libubsan1-armhf-cross
  linux-libc-dev-armhf-cross
Suggested packages:
  binutils-doc gcc-11-locales cpp-doc gcc-11-doc libtool
  gdb-arm-linux-gnueabihf gcc-doc
The following NEW packages will be installed:
  binutils-arm-linux-gnueabihf cpp-11-arm-linux-gnueabihf
  cpp-arm-linux-gnueabihf gcc-11-arm-linux-gnueabihf
  gcc-11-arm-lin

### Build the arm target binary

In [ ]:
!make clean
!make vulnerable-arm

rm -f vulnerable
rm -f vulnerable-arm
arm-linux-gnueabihf-gcc -g -w  -static vulnerable.c -o vulnerable-arm


### The symbols can be retrieved from the binary

In [ ]:
!nm --print-size ./vulnerable-arm | grep -E " main$"
!arm-linux-gnueabihf-objdump -d ./vulnerable-arm
!arm-linux-gnueabihf-objdump --disassemble=main ./vulnerable-arm
!arm-linux-gnueabihf-objdump --disassemble=main ./vulnerable-arm | grep -E "pop.*pc" | cut -f 1 | sed s/://

Streaming output truncated to the last 5000 lines.
   4c5c0:	f8d7 800c 	ldr.w	r8, [r7, #12]
   4c5c4:	4683      	mov	fp, r0
   4c5c6:	2800      	cmp	r0, #0
   4c5c8:	f000 8149 	beq.w	4c85e <_dl_map_object_deps+0x642>
   4c5cc:	697b      	ldr	r3, [r7, #20]
   4c5ce:	6818      	ldr	r0, [r3, #0]
   4c5d0:	693b      	ldr	r3, [r7, #16]
   4c5d2:	4298      	cmp	r0, r3
   4c5d4:	d001      	beq.n	4c5da <_dl_map_object_deps+0x3be>
   4c5d6:	f7d1 fec5 	bl	1e364 <__free>
   4c5da:	4bb1      	ldr	r3, [pc, #708]	; (4c8a0 <_dl_map_object_deps+0x684>)
   4c5dc:	69ba      	ldr	r2, [r7, #24]
   4c5de:	447b      	add	r3, pc
   4c5e0:	681b      	ldr	r3, [r3, #0]
   4c5e2:	58d3      	ldr	r3, [r2, r3]
   4c5e4:	69fa      	ldr	r2, [r7, #28]
   4c5e6:	fab3 f383 	clz	r3, r3
   4c5ea:	2a00      	cmp	r2, #0
   4c5ec:	ea4f 1353 	mov.w	r3, r3, lsr #5
   4c5f0:	bf08      	it	eq
   4c5f2:	2300      	moveq	r3, #0
   4c5f4:	2b00      	cmp	r3, #0
   4c5f6:	f040 81c6 	bne.w	4c986 <_dl_map_object_deps+0x76a>
   4c5fa:	6

### Run the fuzzing using the unicorn emulator
The file harness.py contains the code that emulates the execution environment for the target binary
- Maps the memory
- Copies the code in memory
- Hook (stub) the libc functions
- Copies the afl data in the input buffer at each execution

In [ ]:
%%shell
rm -r ./output
source /content/fuzzingenv/bin/activate
afl-fuzz -U -i ./input_corpus -o ./output -- python3 harness.py @@ ./vulnerable-arm
#AFL_NO_UI=1 AFL_DEBUG=1 AFL_DEBUG_CHILD=1 afl-fuzz -U -i ./input_corpus -o ./output -- python3 harness.py @@ ./vulnerable-arm

[+] Enabled environment variable AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES with value 1
[!] WARNING: Potentially mistyped AFL environment variable: USE_AUTH_EPHEM=1, did you mean AFL_USE_AUTH_EPHEM=1?
afl-fuzz++4.36a based on afl by Michal Zalewski and a large online community
[+] AFL++ is maintained by Marc "van Hauser" Heuse, Dominik Maier, Andrea Fioraldi and Heiko "hexcoder" Eißfeldt
[+] AFL++ is open source, get it at https://github.com/AFLplusplus/AFLplusplus
[+] NOTE: AFL++ >= v3 has changed defaults and behaviours - see README.md
[+] No -M/-S set, autoconfiguring for "-S default"
[*] Getting to work...
[+] Using exploration-based constant power schedule (EXPLORE)
[+] Enabled testcache with 50 MB
[+] Generating fuzz data with a length of min=1 max=1048576
[*] Checking core_pattern...

[-] Your system is configured to send core dump notifications to an
    external utility. This will cause issues: there will be an extended delay
    between stumbling upon a crash and having this info